# When Gradients Stop: `detach()` and `torch.no_grad()`

Use these to **freeze** parts of the graph for inference, metrics, or target networks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Normal backward vs detached tensor

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y.detach()  # breaks graph here
w = z * 3
try:
    w.backward()
except RuntimeError as e:
    print(f"Cannot backprop through detach: {type(e).__name__}")

y.backward()
print(f"Gradient on x from y branch: {x.grad.item()}")

## 2. Compare gradient flow with and without detach

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, grad_on in zip(axes, ['with grad', 'detached'], [True, False]):
    x2 = torch.tensor(2.0, requires_grad=True)
    y2 = x2 ** 2
    z2 = y2 if grad_on else y2.detach()
    loss = z2 * 2
    if grad_on:
        loss.backward()
        g = x2.grad.item()
    else:
        g = 0.0
    ax.bar(['∂loss/∂x'], [g], color='steelblue' if grad_on else 'gray')
    ax.set_ylim(0, 10); ax.set_title(title)
plt.suptitle('detach() blocks gradients to x'); plt.tight_layout(); plt.show()

## 3. `torch.no_grad()` for inference

In [ ]:
model = nn.Linear(4, 2)
x = torch.randn(8, 4)
with torch.no_grad():
    out = model(x)
    print(f"inference output shape: {out.shape}, requires_grad: {out.requires_grad}")

## 4. Training vs eval mode — loss with and without grad

In [ ]:
gen_vals = []
with torch.enable_grad():
  x = torch.randn(16, 4, requires_grad=True)
  gen_vals.append(F.mse_loss(x, torch.zeros_like(x)).item())
with torch.no_grad():
  gen_vals.append(F.mse_loss(x, torch.zeros_like(x)).item())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['enable_grad', 'no_grad'], gen_vals, color=['coral', 'lightgray'], edgecolor='black')
ax.set_title('Same forward op — grad context differs'); ax.set_ylabel('MSE loss')
plt.tight_layout(); plt.show()

# requires_grad flags visualization
flags = {'x': True, 'y=x²': True, 'detach(y)': False, 'no_grad block': False}
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(flags.keys()), [1 if v else 0 for v in flags.values()],
        color=['green' if v else 'red' for v in flags.values()])
ax.set_xlim(0, 1.2); ax.set_title('requires_grad flags along the pipeline')
plt.tight_layout(); plt.show()